# Day 3 — Advanced Models & Complete ML Pipeline

Welcome to the final day of our Machine Learning Fundamentals course!

**What we covered so far:**
- **Day 1:** Linear Regression, Polynomial Regression, model evaluation (MSE, R-squared), the bias-variance tradeoff
- **Day 2:** Classification with Logistic Regression and Decision Trees, evaluation metrics (accuracy, precision, recall, F1, confusion matrix, ROC curve)

**What we'll cover today:**
1. **Support Vector Machines (SVM)** — The maximum margin classifier
2. **Random Forest** — Ensemble learning with many decision trees
3. **Gradient Boosting** — Learning from mistakes sequentially
4. **Hyperparameter Tuning** — Grid Search and Randomized Search
5. **Complete ML Pipeline** — End-to-end workflow on a real-world dataset
6. **Grand Model Comparison** — Comparing everything we've learned

By the end of today, you'll be able to build a complete machine learning pipeline from raw data to final evaluation.

---
## Section 1: Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

# Preprocessing & model selection
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, RandomizedSearchCV, StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Models
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc
)

# Datasets
from sklearn.datasets import load_breast_cancer

# Scipy for randomized search distributions
from scipy import stats

# Settings
%matplotlib inline
sns.set_style('darkgrid')
np.random.seed(42)
warnings.filterwarnings('ignore')

print('All imports successful! Ready for Day 3.')

---
## Section 2: Support Vector Machines (SVM)

### The Maximum Margin Classifier — Finding the Widest Gap

A Support Vector Machine finds the **decision boundary** (hyperplane) that maximizes the **margin** — the distance between the boundary and the nearest data points from each class.

**Key ideas:**
- **Support vectors** are the data points closest to the decision boundary. They "support" or define the boundary.
- A **wider margin** generally leads to better generalization on unseen data.
- SVMs can use **kernel functions** to handle non-linearly separable data by projecting it into a higher-dimensional space.

**Common kernels:**
| Kernel | Use Case |
|--------|----------|
| Linear | Linearly separable data, high-dimensional data (e.g., text) |
| RBF (Radial Basis Function) | Non-linear relationships, general purpose |
| Polynomial | Non-linear with polynomial decision boundaries |

**Important hyperparameters:**
- **C** (regularization): Controls the trade-off between a wide margin and correct classification. High C = narrow margin, fewer errors on training data.
- **gamma** (for RBF kernel): Controls how far the influence of a single training example reaches. High gamma = each point has close-range influence only.

In [ ]:
# Load the Breast Cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0 = malignant, 1 = benign

print(f'Dataset shape: {X.shape}')
print(f'Classes: {data.target_names}')
print(f'Class distribution: {np.bincount(y)}')
print(f'\nFirst 5 features: {list(X.columns[:5])}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling — CRITICAL for SVM!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTraining set: {X_train_scaled.shape}')
print(f'Test set: {X_test_scaled.shape}')

In [ ]:
# SVM with LINEAR kernel
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)

y_pred_linear = svm_linear.predict(X_test_scaled)
acc_linear = accuracy_score(y_test, y_pred_linear)

print(f'SVM (Linear Kernel) Accuracy: {acc_linear:.4f}')
print(f'Number of support vectors: {svm_linear.n_support_}')

In [ ]:
# SVM with RBF kernel
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svm_rbf.predict(X_test_scaled)
acc_rbf = accuracy_score(y_test, y_pred_rbf)

print(f'SVM (RBF Kernel) Accuracy: {acc_rbf:.4f}')
print(f'Number of support vectors: {svm_rbf.n_support_}')

In [ ]:
# Compare Linear vs RBF
print('=== Kernel Comparison ===')
print(f'{"Kernel":<12} {"Accuracy":<12} {"Support Vectors"}')
print('-' * 42)
print(f'{"Linear":<12} {acc_linear:<12.4f} {svm_linear.n_support_}')
print(f'{"RBF":<12} {acc_rbf:<12.4f} {svm_rbf.n_support_}')
print()

if acc_linear > acc_rbf:
    print('Linear kernel performs better on this dataset.')
    print('This suggests the data is (approximately) linearly separable.')
elif acc_rbf > acc_linear:
    print('RBF kernel performs better on this dataset.')
    print('This suggests non-linear decision boundaries help.')
else:
    print('Both kernels perform equally well on this dataset.')

In [ ]:
# Visualize decision boundary using first 2 principal components
# We reduce to 2D with PCA so we can plot the decision boundary

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (kernel, title) in enumerate([('linear', 'Linear Kernel'), ('rbf', 'RBF Kernel')]):
    ax = axes[idx]
    
    # Train SVM on 2D PCA data
    svm_2d = SVC(kernel=kernel, random_state=42)
    svm_2d.fit(X_train_pca, y_train)
    
    # Create meshgrid for decision boundary
    x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
    y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    
    # Predict on meshgrid
    Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision boundary
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.contour(xx, yy, Z, colors='k', linewidths=0.5)
    
    # Plot data points
    scatter = ax.scatter(
        X_train_pca[:, 0], X_train_pca[:, 1],
        c=y_train, cmap='coolwarm', edgecolors='k',
        s=40, alpha=0.7
    )
    
    acc_2d = svm_2d.score(X_test_pca, y_test)
    ax.set_title(f'SVM Decision Boundary ({title})\nAccuracy (2D): {acc_2d:.4f}', fontsize=13)
    ax.set_xlabel('First Principal Component', fontsize=11)
    ax.set_ylabel('Second Principal Component', fontsize=11)

plt.tight_layout()
plt.suptitle('SVM Decision Boundaries (PCA-Reduced to 2D)', fontsize=15, y=1.02)
plt.show()

print('\nNote: Accuracy is lower in 2D because we discarded information by reducing from 30 features to 2.')

In [ ]:
# Effect of C parameter (regularization strength)
# C controls the penalty for misclassification:
#   Small C -> wider margin, more misclassifications allowed (more regularization)
#   Large C -> narrower margin, fewer misclassifications allowed (less regularization)

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_accs_c = []
test_accs_c = []

for C in C_values:
    svm_c = SVC(kernel='rbf', C=C, random_state=42)
    svm_c.fit(X_train_scaled, y_train)
    train_accs_c.append(svm_c.score(X_train_scaled, y_train))
    test_accs_c.append(svm_c.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 6))
plt.plot(C_values, train_accs_c, 'o-', label='Training Accuracy', linewidth=2, markersize=8)
plt.plot(C_values, test_accs_c, 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.xscale('log')
plt.xlabel('C (Regularization Parameter)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of C Parameter on SVM (RBF Kernel)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_c_idx = np.argmax(test_accs_c)
print(f'Best C value: {C_values[best_c_idx]} (Test Accuracy: {test_accs_c[best_c_idx]:.4f})')
print('\nNotice: Very small C underfits, very large C may overfit.')

In [ ]:
# Effect of gamma parameter (RBF kernel width)
# gamma controls how far the influence of a single point reaches:
#   Small gamma -> far reach, smoother boundary (underfitting risk)
#   Large gamma -> close reach, complex boundary (overfitting risk)

gamma_values = [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
train_accs_g = []
test_accs_g = []

for gamma in gamma_values:
    svm_g = SVC(kernel='rbf', gamma=gamma, random_state=42)
    svm_g.fit(X_train_scaled, y_train)
    train_accs_g.append(svm_g.score(X_train_scaled, y_train))
    test_accs_g.append(svm_g.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 6))
plt.plot(gamma_values, train_accs_g, 'o-', label='Training Accuracy', linewidth=2, markersize=8)
plt.plot(gamma_values, test_accs_g, 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.xscale('log')
plt.xlabel('Gamma', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of Gamma Parameter on SVM (RBF Kernel)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_g_idx = np.argmax(test_accs_g)
print(f'Best gamma value: {gamma_values[best_g_idx]} (Test Accuracy: {test_accs_g[best_g_idx]:.4f})')
print('\nNotice: Very large gamma causes severe overfitting -- perfect training score but poor test score.')

In [ ]:
# What happens WITHOUT scaling? Let's see the dramatic difference.

# Without scaling
svm_noscale = SVC(kernel='rbf', random_state=42)
svm_noscale.fit(X_train, y_train)  # using raw, unscaled data
acc_noscale = svm_noscale.score(X_test, y_test)

# With scaling (already computed)
acc_scaled = acc_rbf

print('=== The Importance of Feature Scaling for SVM ===')
print(f'SVM (RBF) WITHOUT scaling: {acc_noscale:.4f}')
print(f'SVM (RBF) WITH scaling:    {acc_scaled:.4f}')
print(f'Difference:                {acc_scaled - acc_noscale:.4f}')
print()
print('Why such a big difference?')
print('SVM computes distances between data points. When features have')
print('very different scales (e.g., one ranges 0-1, another 0-1000),')
print('the large-scale feature dominates the distance calculation.')
print('Scaling puts all features on equal footing.')
print()
print(f'Feature range example (unscaled):')
print(f'  {X.columns[0]}: [{X.iloc[:, 0].min():.2f}, {X.iloc[:, 0].max():.2f}]')
print(f'  {X.columns[20]}: [{X.iloc[:, 20].min():.2f}, {X.iloc[:, 20].max():.2f}]')

### When to Use SVM

**Use SVM when:**
- You have a medium-sized dataset (SVMs don't scale well to very large datasets)
- The number of features is large relative to the number of samples (e.g., text classification)
- You want a strong, well-regularized classifier with a clear theoretical foundation
- Non-linear boundaries are needed (use RBF kernel)

**Avoid SVM when:**
- Your dataset is very large (>100K samples) -- training time grows rapidly
- You need probability estimates (SVM doesn't natively produce them, though `probability=True` adds them via Platt scaling)
- Interpretability is crucial -- SVMs are harder to explain than trees

**Remember:** Always scale your features before using SVM!

---
## Section 3: Random Forest

### The Power of Many Trees — Ensemble Learning

A **Random Forest** builds many decision trees and combines their predictions through **majority voting** (classification) or **averaging** (regression).

**How it works:**
1. Create `n_estimators` bootstrap samples (random samples with replacement)
2. For each sample, grow a decision tree, but at each split only consider a random subset of features
3. For prediction, each tree votes and the majority wins

**Why does this work?**
- Individual trees are "weak" and may overfit, but their errors are different (decorrelated)
- Averaging many decorrelated predictions reduces variance without increasing bias
- This is the core idea behind **bagging** (Bootstrap AGGregating)

Think of it like asking 100 people for directions -- each might be slightly wrong, but the consensus is usually right!

In [ ]:
# Fit a Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f'Random Forest Accuracy: {acc_rf:.4f}')

In [ ]:
# Compare Random Forest with a single Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)
acc_dt = accuracy_score(y_test, y_pred_dt)

print('=== Single Tree vs Forest ===')
print(f'Decision Tree Accuracy:  {acc_dt:.4f}')
print(f'Random Forest Accuracy:  {acc_rf:.4f}')
print(f'Improvement:             {acc_rf - acc_dt:+.4f}')
print()
print('The forest almost always outperforms a single tree!')
print(f'Decision Tree depth: {dt.get_depth()}, leaves: {dt.get_n_leaves()}')
print(f'Random Forest: {rf.n_estimators} trees')

In [ ]:
# Feature importance -- which features matter most?
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

# Plot top 10
plt.figure(figsize=(10, 6))
top10 = feature_importance_df.head(10)
sns.barplot(x='Importance', y='Feature', data=top10, palette='viridis')
plt.title('Random Forest: Top 10 Feature Importances', fontsize=14)
plt.xlabel('Importance (Mean Decrease in Impurity)', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
for i, row in feature_importance_df.head(5).iterrows():
    print(f'  {row["Feature"]}: {row["Importance"]:.4f}')

In [ ]:
# Effect of n_estimators (number of trees)
n_estimator_values = list(range(10, 310, 10))
train_accs_ne = []
test_accs_ne = []

for n in n_estimator_values:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_n.fit(X_train_scaled, y_train)
    train_accs_ne.append(rf_n.score(X_train_scaled, y_train))
    test_accs_ne.append(rf_n.score(X_test_scaled, y_test))

plt.figure(figsize=(10, 6))
plt.plot(n_estimator_values, train_accs_ne, 'o-', label='Training Accuracy',
         linewidth=2, markersize=4)
plt.plot(n_estimator_values, test_accs_ne, 's-', label='Test Accuracy',
         linewidth=2, markersize=4)
plt.xlabel('Number of Trees (n_estimators)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Random Forest: Effect of Number of Trees', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Notice: Adding more trees generally helps, but with diminishing returns.')
print('After a certain point, more trees just increase computation without improving accuracy.')

In [ ]:
# Out-of-Bag (OOB) Score demonstration
# Each tree is trained on a bootstrap sample (~63% of data).
# The remaining ~37% (out-of-bag samples) are used for internal validation.
# This gives us a free validation score without needing a separate validation set!

rf_oob = RandomForestClassifier(
    n_estimators=100, oob_score=True, random_state=42
)
rf_oob.fit(X_train_scaled, y_train)

print('=== Out-of-Bag (OOB) Score ===')
print(f'OOB Score:       {rf_oob.oob_score_:.4f}')
print(f'Test Accuracy:   {rf_oob.score(X_test_scaled, y_test):.4f}')
print()
print('The OOB score approximates the test accuracy without needing a separate')
print('validation set. Each sample is predicted only by trees that did NOT')
print('include it in their bootstrap sample.')
print()
print('This is one of the nice built-in features of Random Forest!')

### Why Random Forest Reduces Overfitting

A single decision tree will often **overfit** by memorizing noise in the training data. Random Forest combats this through **two sources of randomness**:

1. **Bootstrap sampling**: Each tree sees a different subset of the training data
2. **Random feature selection**: At each split, only a random subset of features is considered

These two mechanisms ensure that the trees are **diverse** -- they make different errors. When we average their predictions, the individual errors cancel out, leaving a more robust prediction.

**Mathematically:**
- If each tree has variance sigma-squared and the correlation between trees is rho, then the variance of the forest is:
  - `Var_forest = rho * sigma^2 + (1 - rho) / n * sigma^2`
- As we add more trees (n -> infinity), the second term vanishes, and variance is determined by rho
- Lower correlation between trees = lower variance = less overfitting

---
## Section 4: Gradient Boosting

### Learning from Mistakes — Sequential Correction

While Random Forest builds trees **independently** (in parallel), Gradient Boosting builds trees **sequentially**, where each new tree tries to correct the errors of the previous ones.

**How it works:**
1. Start with a simple prediction (e.g., the most common class)
2. Compute the residuals (errors) of the current model
3. Fit a small tree to predict these residuals
4. Add this tree's predictions (scaled by a learning rate) to the current model
5. Repeat steps 2-4

**Key hyperparameters:**
- **n_estimators**: Number of sequential trees (boosting rounds)
- **learning_rate**: How much each tree contributes. Smaller = more trees needed but often better results
- **max_depth**: Depth of each individual tree (usually shallow, like 3-5)

In [ ]:
# Fit a Gradient Boosting Classifier
gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)
gb.fit(X_train_scaled, y_train)

y_pred_gb = gb.predict(X_test_scaled)
acc_gb = accuracy_score(y_test, y_pred_gb)

print(f'Gradient Boosting Accuracy: {acc_gb:.4f}')

In [ ]:
# Compare Gradient Boosting with Random Forest
print('=== Random Forest vs Gradient Boosting ===')
print(f'Random Forest Accuracy:      {acc_rf:.4f}')
print(f'Gradient Boosting Accuracy:  {acc_gb:.4f}')
print()

if acc_gb > acc_rf:
    print('Gradient Boosting edges ahead here, likely because it focuses')
    print('on correcting mistakes that Random Forest misses.')
elif acc_rf > acc_gb:
    print('Random Forest wins here. On some datasets, the bagging approach')
    print('generalizes better than the sequential correction of boosting.')
else:
    print('Both perform equally well on this dataset!')

In [ ]:
# Plot training deviance -- how error decreases with each boosting round
# staged_predict gives predictions at each stage (after each tree is added)

# Compute deviance at each stage for train and test
train_deviance = np.zeros(gb.n_estimators)
test_deviance = np.zeros(gb.n_estimators)

for i, (y_train_pred, y_test_pred) in enumerate(
    zip(gb.staged_predict(X_train_scaled), gb.staged_predict(X_test_scaled))
):
    train_deviance[i] = 1 - accuracy_score(y_train, y_train_pred)
    test_deviance[i] = 1 - accuracy_score(y_test, y_test_pred)

plt.figure(figsize=(10, 6))
plt.plot(range(1, gb.n_estimators + 1), train_deviance, label='Training Error',
         linewidth=2)
plt.plot(range(1, gb.n_estimators + 1), test_deviance, label='Test Error',
         linewidth=2)
plt.xlabel('Number of Boosting Rounds', fontsize=12)
plt.ylabel('Error Rate', fontsize=12)
plt.title('Gradient Boosting: Training vs Test Error Over Rounds', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Training error consistently decreases as more trees are added.')
print('Test error decreases then may level off or slightly increase (overfitting).')
print('This plot helps you find the optimal number of boosting rounds.')

In [ ]:
# Effect of learning rate
# Smaller learning rate = each tree contributes less, needs more trees
# Often produces better results but takes longer to train

learning_rates = [0.01, 0.1, 0.5, 1.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, lr in enumerate(learning_rates):
    ax = axes[idx // 2, idx % 2]
    
    gb_lr = GradientBoostingClassifier(
        n_estimators=200, learning_rate=lr, max_depth=3, random_state=42
    )
    gb_lr.fit(X_train_scaled, y_train)
    
    # Track error at each stage
    train_err = []
    test_err = []
    for y_tr_pred, y_te_pred in zip(
        gb_lr.staged_predict(X_train_scaled),
        gb_lr.staged_predict(X_test_scaled)
    ):
        train_err.append(1 - accuracy_score(y_train, y_tr_pred))
        test_err.append(1 - accuracy_score(y_test, y_te_pred))
    
    ax.plot(range(1, 201), train_err, label='Train Error', linewidth=1.5)
    ax.plot(range(1, 201), test_err, label='Test Error', linewidth=1.5)
    ax.set_title(f'Learning Rate = {lr}', fontsize=12)
    ax.set_xlabel('Boosting Rounds', fontsize=10)
    ax.set_ylabel('Error Rate', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_ylim(-0.01, 0.15)

plt.suptitle('Effect of Learning Rate on Gradient Boosting', fontsize=15)
plt.tight_layout()
plt.show()

print('Observations:')
print('- lr=0.01: Slow convergence, needs many rounds. Less risk of overfitting.')
print('- lr=0.1:  Good balance -- converges reasonably fast with stable test error.')
print('- lr=0.5:  Fast convergence but may start overfitting sooner.')
print('- lr=1.0:  Very aggressive -- high risk of overfitting and instability.')

In [ ]:
# Feature importance comparison: Random Forest vs Gradient Boosting

rf_importance = pd.DataFrame({
    'Feature': data.feature_names,
    'RF_Importance': rf.feature_importances_
}).sort_values('RF_Importance', ascending=False).head(10)

gb_importance = pd.DataFrame({
    'Feature': data.feature_names,
    'GB_Importance': gb.feature_importances_
}).sort_values('GB_Importance', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest importances
sns.barplot(x='RF_Importance', y='Feature', data=rf_importance,
            palette='Blues_r', ax=axes[0])
axes[0].set_title('Random Forest: Top 10 Features', fontsize=13)
axes[0].set_xlabel('Importance', fontsize=11)

# Gradient Boosting importances
sns.barplot(x='GB_Importance', y='Feature', data=gb_importance,
            palette='Oranges_r', ax=axes[1])
axes[1].set_title('Gradient Boosting: Top 10 Features', fontsize=13)
axes[1].set_xlabel('Importance', fontsize=11)

plt.suptitle('Feature Importance Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('The two methods may rank features differently!')
print('Random Forest importance = mean decrease in impurity across all trees.')
print('Gradient Boosting importance = how often a feature is used to make key splits.')

### Random Forest vs Gradient Boosting -- When to Use Which

| Aspect | Random Forest | Gradient Boosting |
|--------|--------------|-------------------|
| **Training** | Trees built independently (parallelizable) | Trees built sequentially (slower) |
| **Overfitting** | Resistant -- more trees rarely hurts | Can overfit if too many rounds |
| **Hyperparameters** | Fewer to tune | More to tune (learning_rate, n_estimators, depth) |
| **Performance** | Good "out of the box" | Often higher accuracy when well-tuned |
| **Interpretability** | Feature importance available | Feature importance available |
| **Speed** | Faster training (parallel trees) | Slower training (sequential) |

**Rule of thumb:**
- Start with **Random Forest** -- it's harder to mess up
- Try **Gradient Boosting** when you want to squeeze out extra performance and have time to tune

---
## Section 5: Hyperparameter Tuning

### Finding the Best Settings — Grid Search and Randomized Search

So far we've been using default hyperparameters or manually trying a few values. In practice, we need a systematic approach to find the best combination.

**Two main approaches:**

1. **Grid Search (`GridSearchCV`)**: Try every possible combination of specified parameter values. Exhaustive but can be slow.

2. **Randomized Search (`RandomizedSearchCV`)**: Sample random combinations from parameter distributions. Faster and often finds good solutions with fewer evaluations.

Both use **cross-validation** internally to evaluate each parameter combination, so you get an honest estimate of performance.

In [ ]:
# GridSearchCV on Random Forest
# We define a grid of hyperparameters to try ALL combinations

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 2, 5]
}

# Total combinations: 3 x 4 x 3 = 36, each evaluated with 5-fold CV = 180 fits
print(f'Total parameter combinations: {3 * 4 * 3}')
print(f'Total model fits (with 5-fold CV): {3 * 4 * 3 * 5}')
print()

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=0
)

start_time = time.time()
grid_search_rf.fit(X_train_scaled, y_train)
grid_time = time.time() - start_time

print(f'Grid Search completed in {grid_time:.2f} seconds')
print(f'\nBest Parameters: {grid_search_rf.best_params_}')
print(f'Best CV Score:   {grid_search_rf.best_score_:.4f}')
print(f'Test Score:      {grid_search_rf.score(X_test_scaled, y_test):.4f}')

In [ ]:
# RandomizedSearchCV on Gradient Boosting
# Instead of trying all combinations, we sample from distributions

param_distributions_gb = {
    'n_estimators': stats.randint(50, 300),
    'learning_rate': stats.uniform(0.01, 0.49),  # range: 0.01 to 0.50
    'max_depth': stats.randint(2, 8),
    'min_samples_leaf': stats.randint(1, 10),
    'subsample': stats.uniform(0.6, 0.4)  # range: 0.6 to 1.0
}

random_search_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_distributions=param_distributions_gb,
    n_iter=50,  # Try 50 random combinations
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

start_time = time.time()
random_search_gb.fit(X_train_scaled, y_train)
random_time = time.time() - start_time

print(f'Randomized Search completed in {random_time:.2f} seconds')
print(f'\nBest Parameters: {random_search_gb.best_params_}')
print(f'Best CV Score:   {random_search_gb.best_score_:.4f}')
print(f'Test Score:      {random_search_gb.score(X_test_scaled, y_test):.4f}')

In [ ]:
# Visualize Grid Search results as a heatmap
# Let's look at mean score for n_estimators vs max_depth

results = pd.DataFrame(grid_search_rf.cv_results_)

# Pivot to create heatmap data (average over min_samples_leaf)
pivot_data = results.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators',
    aggfunc='mean'
)

plt.figure(figsize=(10, 6))
sns.heatmap(
    pivot_data, annot=True, fmt='.4f', cmap='YlOrRd',
    linewidths=0.5, cbar_kws={'label': 'Mean CV Accuracy'}
)
plt.title('Grid Search Results: Mean CV Accuracy\n(n_estimators vs max_depth, averaged over min_samples_leaf)',
          fontsize=13)
plt.xlabel('n_estimators', fontsize=12)
plt.ylabel('max_depth', fontsize=12)
plt.tight_layout()
plt.show()

print('Darker/brighter cells indicate better performance.')
print('This visualization helps identify which parameter regions work best.')

### When to Use Grid Search vs Randomized Search

| Aspect | Grid Search | Randomized Search |
|--------|-------------|-------------------|
| **Approach** | Exhaustive -- tries all combinations | Stochastic -- samples random combos |
| **Speed** | Slow for large grids | Much faster |
| **Coverage** | Complete coverage of the grid | Can explore a wider range |
| **Best for** | Small parameter spaces | Large parameter spaces |
| **Guarantees** | Finds the best in the grid | May miss the best, but often close |

**Practical advice:**
- Use **Grid Search** when you have 2-3 parameters with a few values each
- Use **Randomized Search** when you have many parameters or continuous ranges
- A common strategy: start with Randomized Search to narrow down the range, then refine with Grid Search

---
## Section 6: Complete ML Pipeline

### Putting It All Together — End-to-End Workflow

Now let's apply everything we've learned to a **realistic, messy dataset** -- the Titanic survival dataset. This is the kind of data you'll encounter in the real world: missing values, mixed data types, categorical variables, and features that need engineering.

**Our pipeline will follow these steps:**
1. Explore the data
2. Clean and preprocess
3. Train/test split
4. Feature scaling
5. Train multiple models
6. Compare with cross-validation
7. Hyperparameter tune the best model
8. Final evaluation on the test set
9. Wrap it in an sklearn Pipeline

In [ ]:
# Load the Titanic dataset -- a realistic, messy dataset
titanic = sns.load_dataset('titanic')

print(f'Dataset shape: {titanic.shape}')
print(f'\nFirst few rows:')
titanic.head()

In [ ]:
# Step 1 -- Explore the data
print('=== Step 1: Data Exploration ===')
print(f'\nShape: {titanic.shape}')
print(f'\n--- Data Types ---')
print(titanic.dtypes)
print(f'\n--- Missing Values ---')
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
print(missing_df[missing_df['Missing'] > 0])
print(f'\n--- Statistical Summary ---')
titanic.describe()

In [ ]:
# Step 2 -- Clean and preprocess
print('=== Step 2: Data Cleaning & Preprocessing ===')

# Make a copy to avoid modifying the original
df = titanic.copy()

# Handle missing values
# Age: fill with median (robust to outliers)
df['age'].fillna(df['age'].median(), inplace=True)
print(f'Age: filled {titanic["age"].isnull().sum()} missing values with median ({df["age"].median():.0f})')

# Embarked: fill with mode (most common port)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
print(f'Embarked: filled {titanic["embarked"].isnull().sum()} missing values with mode ("{df["embarked"].mode()[0]}")')

# Deck (cabin): too many missing values (>77%), drop it
df.drop(columns=['deck'], inplace=True)
print('Deck: dropped (>77% missing)')

# Drop columns that won't be useful or are redundant
df.drop(columns=['alive', 'who', 'adult_male', 'class', 'embark_town', 'alone'], inplace=True)
print('Dropped redundant columns: alive, who, adult_male, class, embark_town, alone')

# Encode categorical variables
# Sex: male=1, female=0
df['sex'] = LabelEncoder().fit_transform(df['sex'])

# Embarked: one-hot encoding
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

print(f'\nCleaned dataset shape: {df.shape}')
print(f'Missing values remaining: {df.isnull().sum().sum()}')
print(f'\nFeatures: {list(df.columns)}')
df.head()

In [ ]:
# Step 3 -- Train/test split (stratified to preserve class balance)
print('=== Step 3: Train/Test Split ===')

# Separate features and target
X_titanic = df.drop(columns=['survived'])
y_titanic = df['survived']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic
)

print(f'Training set: {X_tr.shape} | Survival rate: {y_tr.mean():.2%}')
print(f'Test set:     {X_te.shape} | Survival rate: {y_te.mean():.2%}')
print(f'\nStratification preserved the class balance in both sets.')

In [ ]:
# Step 4 -- Feature scaling (fit on train only!)
print('=== Step 4: Feature Scaling ===')

scaler_titanic = StandardScaler()
X_tr_scaled = scaler_titanic.fit_transform(X_tr)   # fit + transform on training
X_te_scaled = scaler_titanic.transform(X_te)        # only transform on test

print('StandardScaler fitted on training data, then applied to both sets.')
print()
print('IMPORTANT: We fit the scaler ONLY on training data!')
print('If we fit on the full dataset, we would leak information from the test set')
print('into our preprocessing -- this is called "data leakage".')

In [ ]:
# Step 5 -- Train multiple models
print('=== Step 5: Training Multiple Models ===')

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42, probability=True),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results_step5 = []

for name, model in models.items():
    start = time.time()
    model.fit(X_tr_scaled, y_tr)
    train_time = time.time() - start
    
    y_pred = model.predict(X_te_scaled)
    acc = accuracy_score(y_te, y_pred)
    
    results_step5.append({
        'Model': name,
        'Test Accuracy': acc,
        'Train Time (s)': train_time
    })
    print(f'{name:<22} Accuracy: {acc:.4f}  (trained in {train_time:.3f}s)')

results_df = pd.DataFrame(results_step5).sort_values('Test Accuracy', ascending=False)
print(f'\nBest model: {results_df.iloc[0]["Model"]}')

In [ ]:
# Step 6 -- Compare with cross-validation
# Single train/test split can be misleading. Cross-validation gives a more robust estimate.

print('=== Step 6: Cross-Validation Comparison ===')

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X_tr_scaled, y_tr, cv=5, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'Mean CV Score': scores.mean(),
        'Std': scores.std()
    })
    print(f'{name:<22} CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')

cv_df = pd.DataFrame(cv_results).sort_values('Mean CV Score', ascending=False)

# Bar chart with error bars
plt.figure(figsize=(10, 6))
bars = plt.barh(
    cv_df['Model'], cv_df['Mean CV Score'],
    xerr=cv_df['Std'], color=sns.color_palette('viridis', len(cv_df)),
    capsize=5, edgecolor='black', linewidth=0.5
)
plt.xlabel('Mean CV Accuracy', fontsize=12)
plt.title('Model Comparison: 5-Fold Cross-Validation', fontsize=14)
plt.xlim(0.70, 0.90)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

best_model_name = cv_df.iloc[0]['Model']
print(f'\nBest model by cross-validation: {best_model_name}')

In [ ]:
# Step 7 -- Hyperparameter tune the best model
# We'll tune the best-performing model from cross-validation

print('=== Step 7: Hyperparameter Tuning ===')
print(f'Tuning: {best_model_name}')
print()

# Define parameter grid based on which model won
tuning_configs = {
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [2, 3, 5]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_leaf': [1, 2, 5]
        }
    },
    'SVM (RBF)': {
        'model': SVC(kernel='rbf', random_state=42, probability=True),
        'params': {
            'C': [0.1, 1, 10],
            'gamma': ['scale', 'auto', 0.01, 0.1]
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [3, 5, 7, 10, None],
            'min_samples_leaf': [1, 2, 5, 10]
        }
    }
}

config = tuning_configs[best_model_name]

tuner = GridSearchCV(
    config['model'],
    param_grid=config['params'],
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
tuner.fit(X_tr_scaled, y_tr)

print(f'Best Parameters: {tuner.best_params_}')
print(f'Best CV Score:   {tuner.best_score_:.4f}')
print(f'Test Score:      {tuner.score(X_te_scaled, y_te):.4f}')

best_model = tuner.best_estimator_

In [ ]:
# Step 8 -- Final evaluation on the test set
print('=== Step 8: Final Evaluation ===')

y_final_pred = best_model.predict(X_te_scaled)

# Classification Report
print('Classification Report:')
print(classification_report(y_te, y_final_pred, target_names=['Did not survive', 'Survived']))

# Confusion Matrix Heatmap and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_te, y_final_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Did not survive', 'Survived'],
    yticklabels=['Did not survive', 'Survived'],
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix', fontsize=13)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].set_xlabel('Predicted', fontsize=11)

# ROC Curve
if hasattr(best_model, 'predict_proba'):
    y_proba = best_model.predict_proba(X_te_scaled)[:, 1]
elif hasattr(best_model, 'decision_function'):
    y_proba = best_model.decision_function(X_te_scaled)
else:
    y_proba = y_final_pred  # fallback

fpr, tpr, _ = roc_curve(y_te, y_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', linewidth=2,
             label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=13)
axes[1].legend(fontsize=10)

plt.suptitle(f'Final Model: {best_model_name}', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Step 9 -- Using sklearn Pipeline
# A Pipeline chains preprocessing and modeling into a single object.
# This prevents data leakage and makes deployment easier.

print('=== Step 9: sklearn Pipeline ===')
print()

# Create a pipeline that combines scaling and the best model
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
    ))
])

# Now we can fit and predict in one step -- no manual scaling!
pipe.fit(X_tr, y_tr)  # Fits scaler AND model on raw training data
pipe_score = pipe.score(X_te, y_te)

# Cross-validation with pipeline -- scaling is done inside each fold!
pipe_cv_scores = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring='accuracy')

print('Pipeline structure:')
print(pipe)
print(f'\nPipeline Test Accuracy: {pipe_score:.4f}')
print(f'Pipeline CV Accuracy:   {pipe_cv_scores.mean():.4f} (+/- {pipe_cv_scores.std():.4f})')
print()
print('Benefits of using a Pipeline:')
print('1. Prevents data leakage -- scaling is done inside each CV fold')
print('2. Cleaner code -- fit/predict in one call')
print('3. Easy to deploy -- save one object that handles everything')
print('4. Works with GridSearchCV -- tune scaler + model parameters together')

### Summary of the Full ML Workflow

Here's the complete pipeline we just built:

```
Raw Data
  |-> Explore (shape, types, missing values, distributions)
  |-> Clean (handle missing, encode categoricals, select features)
  |-> Split (train/test, stratified)
  |-> Scale (fit on train only!)
  |-> Train multiple models
  |-> Compare with cross-validation
  |-> Tune the best model's hyperparameters
  |-> Final evaluation (confusion matrix, ROC curve, classification report)
  |-> Package in an sklearn Pipeline
```

This is the workflow you should follow for any classification problem. The specific models and preprocessing steps will vary, but the overall structure remains the same.

---
## Section 7: Grand Model Comparison

### Comparing ALL Models We've Learned Across 3 Days

Let's bring together every model from our course and compare them on the Titanic dataset.

In [ ]:
# Comprehensive comparison: all models, multiple metrics, training time

all_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM (Linear)': SVC(kernel='linear', random_state=42, probability=True),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42, probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

comparison = []

for name, model in all_models.items():
    # Train and time
    start = time.time()
    model.fit(X_tr_scaled, y_tr)
    train_time = time.time() - start
    
    # Predict
    y_pred = model.predict(X_te_scaled)
    
    # Metrics
    comparison.append({
        'Model': name,
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred),
        'Recall': recall_score(y_te, y_pred),
        'F1 Score': f1_score(y_te, y_pred),
        'Train Time (s)': round(train_time, 4)
    })

comparison_df = pd.DataFrame(comparison).sort_values('F1 Score', ascending=False)
comparison_df.index = range(1, len(comparison_df) + 1)

print('=== Grand Model Comparison on Titanic Dataset ===')
print()
print(comparison_df.to_string())

In [ ]:
# Grouped bar chart comparing models across metrics

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
model_names = comparison_df['Model'].tolist()

x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for i, metric in enumerate(metrics):
    values = comparison_df[metric].tolist()
    ax.bar(x + i * width, values, width, label=metric, color=colors[i],
           edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Grand Model Comparison: All Metrics', fontsize=15)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Training time comparison
plt.figure(figsize=(10, 4))
plt.barh(comparison_df['Model'], comparison_df['Train Time (s)'],
         color=sns.color_palette('coolwarm', len(comparison_df)),
         edgecolor='black', linewidth=0.5)
plt.xlabel('Training Time (seconds)', fontsize=12)
plt.title('Training Time Comparison', fontsize=14)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Key Insights -- The No Free Lunch Theorem

The **No Free Lunch Theorem** states that no single algorithm is best for every problem. Each model has strengths and weaknesses:

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Logistic Regression** | Fast, interpretable, good baseline | Assumes linear decision boundary |
| **Decision Tree** | Interpretable, handles non-linearity | Prone to overfitting |
| **SVM** | Powerful with kernels, works in high dimensions | Slow on large data, needs scaling |
| **Random Forest** | Robust, resistant to overfitting, few hyperparameters | Less interpretable than single tree |
| **Gradient Boosting** | Often highest accuracy when tuned | Sensitive to hyperparameters, slower |

**The practical takeaway:**
- Always try multiple models
- Use cross-validation to compare honestly
- Tune the most promising candidates
- Consider not just accuracy, but also training time, interpretability, and the specific metric that matters for your problem

---
## Section 8: Practice Exercises

### Final Challenge -- Try It Yourself

Now it's your turn! Apply everything you've learned over the past 3 days to solve these challenges.

---

**Exercise 1: Complete Pipeline on the Wine Dataset**

Load the Wine dataset from sklearn (`from sklearn.datasets import load_wine`). Build a complete pipeline:
1. Preprocess the data (scaling)
2. Train at least 5 different models (Logistic Regression, Decision Tree, SVM, Random Forest, Gradient Boosting)
3. Compare using 5-fold cross-validation
4. Tune the best model with GridSearchCV
5. Evaluate on the test set with a confusion matrix and classification report

---

**Exercise 2: Best Model for Digits**

Load the Digits dataset (`from sklearn.datasets import load_digits`). This is a 10-class classification problem (handwritten digits 0-9).
- Which model performs best? Use 5-fold cross-validation to find out.
- Does scaling matter for all models? Test with and without scaling.
- Plot the confusion matrix for the best model -- which digits get confused with each other?

---

**Exercise 3: sklearn Pipeline with PCA**

Create an sklearn `Pipeline` that combines:
1. `StandardScaler`
2. `PCA(n_components=10)`
3. `RandomForestClassifier`

Apply it to the Digits dataset and evaluate with 5-fold cross-validation. Then compare it with a pipeline that does NOT use PCA. Does dimensionality reduction help or hurt?

```python
# Hint:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
```

---

**Exercise 4: Regression Pipeline**

Build a regression pipeline for the California Housing dataset (`from sklearn.datasets import fetch_california_housing`):
1. Load and explore the data
2. Train three regressors: `SVR`, `RandomForestRegressor`, and `GradientBoostingRegressor`
3. Compare using cross-validation with `scoring='neg_mean_squared_error'`
4. Tune the best model
5. Evaluate with MSE and R-squared on the test set

```python
# Hint:
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.datasets import fetch_california_housing
```

---
## Section 9: Course Summary

### What We've Learned Across 3 Days

| Day | Topic | Models & Concepts |
|-----|-------|-------------------|
| **Day 1** | Regression | Linear Regression, Polynomial Regression, MSE, R-squared, bias-variance tradeoff |
| **Day 2** | Classification & Evaluation | Logistic Regression, Decision Trees, accuracy, precision, recall, F1, confusion matrix, ROC curve |
| **Day 3** | Advanced Models & Pipeline | SVM (linear/RBF, C, gamma), Random Forest (bagging, feature importance, OOB), Gradient Boosting (learning rate, staged predictions), hyperparameter tuning (Grid/Randomized Search), complete ML pipeline, sklearn Pipeline |

### Key Principles to Remember

1. **Always split your data** before any preprocessing (train/test split)
2. **Scale features** when using distance-based algorithms (SVM, KNN) or gradient-based optimization (Logistic Regression, neural nets)
3. **Use cross-validation** to get reliable performance estimates
4. **Try multiple models** -- no single algorithm is best for every problem
5. **Tune hyperparameters** systematically, not by guessing
6. **Use sklearn Pipeline** to prevent data leakage and keep code clean
7. **Choose the right metric** for your problem -- accuracy isn't always enough
8. **Understand the tradeoffs** -- accuracy vs interpretability vs training time

### Resources for Further Learning

- **scikit-learn documentation**: https://scikit-learn.org/stable/ (excellent examples and user guide)
- **"Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow"** by Aurelien Geron (comprehensive textbook)
- **"An Introduction to Statistical Learning (ISLR)"** by James, Witten, Hastie, and Tibshirani (free PDF, great for theory)
- **Kaggle**: https://www.kaggle.com (practice with real datasets and competitions)
- **StatQuest with Josh Starmer** on YouTube (clear visual explanations of ML concepts)

---

**Congratulations on completing the Machine Learning Fundamentals course!**

You now have the foundation to tackle real-world ML problems. The next steps would be to explore:
- Deep Learning (neural networks with TensorFlow/PyTorch)
- Natural Language Processing (NLP)
- Feature engineering techniques
- Model deployment and production pipelines